In [1]:
import os
import warnings

if 'Modeling' in os.path.abspath("").split('/'):
    os.chdir('..')
if 'Notebooks' in os.path.abspath("").split('/'):
    os.chdir('..')

project_root = os.path.abspath("")

warnings.filterwarnings('ignore')

In [2]:
from Scripts.LazyLoadMorletDataset import LazyLoadMorletDataset

In [3]:
# data = LazyLoadMorletDataset('./Generated/Data_Train/', './Generated/Spectrums/exec_morlets', eeg_resampling_freq=256, n_jobs=12, max_eeg_raw_length=15500, max_eog_raw_length=15500)

In [4]:
# data[0]

In [5]:
import os
import json
import traceback
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

In [6]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

DATASET_PATH = Path('./Generated/Data_Train/')
MORLET_PATH = Path('./Generated/Spectrums/exec_morlets')
TASK_TYPE = 'geometric'
RESULT_PATH = Path('./Generated/Results/torch_results.json')

DATASET_CONFIG = {
    'eeg_resampling_freq': 256,
    'max_morlet_wav_len': 309,
    'n_jobs': 12,
    'max_eeg_raw_length' : 15500, 
    'max_eog_raw_length' : 15500,
    'eog_needed': True,
}

TRAIN_CONFIG = {
    'batch_size': 4,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'epochs': 50,
    'early_stop_patience': 10,
    'seed': 42,
}

SPLITS = [3, 5, 7]
METRICS_LIST = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

Using device: cuda


In [7]:
class EEGBranch(nn.Module):
    """Обработка сырой ЭЭГ: 1D-свертки + глобальный пулинг"""
    def __init__(self, in_ch, base_ch=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, base_ch, kernel_size=7, padding=3),
            nn.BatchNorm1d(base_ch),
            nn.ReLU(),
            nn.Conv1d(base_ch, base_ch*2, kernel_size=5, padding=2),
            nn.BatchNorm1d(base_ch*2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(dropout)
        )
        self.out_dim = base_ch * 2

    def forward(self, x):
        # x: [B, channels, time]
        return self.net(x)

In [8]:
class SpectralBranch(nn.Module):
    """Обработка вейвлет-признаков (Power/Phase): FC-слои"""
    def __init__(self, in_features, hidden=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.out_dim = hidden // 2

    def forward(self, x):
        # x: [B, features]
        return self.net(x)


class EIRNet(nn.Module):
    """
    Универсальная модель.
    input_config: dict с флагами {'eeg': bool, 'power': bool, 'phase': bool}
    """
    def __init__(self, 
                 eeg_ch: int, 
                 eeg_time: int,
                 spectral_features: int,
                 n_classes: int,
                 input_config: dict,
                 hidden_dim: int = 64):
        super().__init__()
        self.input_config = input_config
        branches = []
        out_dims = []
        
        if input_config['eeg']:
            self.eeg_branch = EEGBranch(eeg_ch)
            branches.append(self.eeg_branch)
            out_dims.append(self.eeg_branch.out_dim)
        
        if input_config['power']:
            self.power_branch = SpectralBranch(spectral_features)
            branches.append(self.power_branch)
            out_dims.append(self.power_branch.out_dim)
            
        if input_config['phase']:
            self.phase_branch = SpectralBranch(spectral_features)
            branches.append(self.phase_branch)
            out_dims.append(self.phase_branch.out_dim)
        
        self.branches = nn.ModuleList(branches)
        total_dim = sum(out_dims)
        
        self.classifier = nn.Sequential(
            nn.Linear(total_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, inputs):
        if not isinstance(inputs, (list, tuple)):
            inputs = [inputs]
        
        features = []
        branch_idx = 0
        input_idx = 0  # ← Новый счётчик для inputs
        
        if self.input_config['eeg']:
            features.append(self.branches[branch_idx](inputs[input_idx]))
            branch_idx += 1
            input_idx += 1  # ← Инкрементируем только если eeg активен
        
        if self.input_config['power']:
            features.append(self.branches[branch_idx](inputs[input_idx]))
            branch_idx += 1
            input_idx += 1  # ← Инкрементируем только если power активен
        
        if self.input_config['phase']:
            features.append(self.branches[branch_idx](inputs[input_idx]))
            branch_idx += 1
            input_idx += 1  # ← Инкрементируем только если phase активен
        
        x = torch.cat(features, dim=1)
        return self.classifier(x)


In [9]:
def collate_fn_eir(batch, input_type: int, eog_needed: bool):
    """
    Собирает батч из элементов датасета.
    Возвращает: (inputs, labels)
    inputs: Tensor или list of Tensors
    labels: 1D torch.Tensor [B]
    """
    # Распаковка
    if eog_needed:
        eegs, eogs, powers, phases, labels_raw = [], [], [], [], []
        for item in batch:
            eegs.append(item[0])
            eogs.append(item[1])
            powers.append(item[2])
            phases.append(item[3])
            labels_raw.append(item[4])  # Может быть int или torch.Tensor
    else:
        eegs, powers, phases, labels_raw = [], [], [], []
        for item in batch:
            eegs.append(item[0])
            powers.append(item[1])
            phases.append(item[2])
            labels_raw.append(item[3])
    
    # Стек тензоров
    eegs = torch.stack(eegs).float()              # [B, C, T]
    
    # Спектральные признаки: сплющиваем [B, C, F*T] -> [B, C*F*T]
    powers = torch.stack(powers).float()
    powers = powers.view(powers.shape[0], -1)
    
    phases = torch.stack(phases).float()
    phases = phases.view(phases.shape[0], -1)
    
    # === КРИТИЧЕСКИЙ ФИКС для labels ===
    # Извлекаем скалярные значения, даже если в списке уже тензоры
    labels = torch.tensor(
        [lbl.item() if isinstance(lbl, torch.Tensor) else int(lbl) for lbl in labels_raw],
        dtype=torch.long
    )
    
    # Формирование входа под input_type
    if input_type == 1:
        inputs = eegs
    elif input_type == 2:
        inputs = [eegs, powers, phases]
    elif input_type == 3:
        inputs = [eegs, powers]
    elif input_type == 4:
        inputs = [eegs, phases]
    elif input_type == 5:
        inputs = [powers, phases]
    else:
        raise ValueError(f"Unknown input_type: {input_type}")
    
    return inputs, labels

In [10]:
def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_metrics(y_true, y_pred, y_scores=None):
    results = {}
    results['accuracy'] = accuracy_score(y_true, y_pred)
    results['precision'] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    results['recall'] = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    results['f1'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    if y_scores is not None:
        try:
            results['roc_auc'] = roc_auc_score(y_true, y_scores, multi_class='ovr', average='weighted')
        except:
            results['roc_auc'] = -1.0
    return results

def train_epoch(model, loader, criterion, optimizer, device, scaler, input_type, eog_needed):
    model.train()
    total_loss = 0
    for inputs, labels in loader:
        # Перенос на устройство: обрабатываем и тензор, и список тензоров
        if isinstance(inputs, list):
            inputs = [x.to(device, non_blocking=True) for x in inputs]
        else:
            inputs = inputs.to(device, non_blocking=True)
        
        labels = labels.to(device, non_blocking=True)  # Теперь это гарантированно тензор
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu', enabled=(device.type=='cuda')):
            logits = model(inputs)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        del inputs, labels, logits, loss
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device, input_type, eog_needed):
    model.eval()
    all_preds, all_labels, all_scores = [], [], []
    for inputs, labels in loader:
        if isinstance(inputs, list):
            inputs = [x.to(device, non_blocking=True) for x in inputs]
        else:
            inputs = inputs.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu', enabled=(device.type=='cuda')):
            logits = model(inputs)
            probs = torch.softmax(logits, dim=1)
        
        preds = torch.argmax(probs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_scores.extend(probs.cpu().numpy())
    
    return compute_metrics(all_labels, all_preds, np.array(all_scores))

In [11]:
def main():
    set_seed(TRAIN_CONFIG['seed'])
    
    # Инициализация датасета
    print(f"Loading dataset from {DATASET_PATH}...")
    dataset = LazyLoadMorletDataset(
        dataset_path=DATASET_PATH,
        morlet_data_path=MORLET_PATH,
        task_type=TASK_TYPE,
        **DATASET_CONFIG
    )
    print(f"Dataset: {len(dataset)} samples")
    
    # Инференс параметров из первого семпла
    sample = dataset[0]
    if dataset.eog_needed:
        EEG_SAMPLE, _, POWER_SAMPLE, PHASE_SAMPLE, _, _, _ = sample
    else:
        EEG_SAMPLE, POWER_SAMPLE, PHASE_SAMPLE, _, _, _ = sample
    
    EEG_CH, EEG_TIME = EEG_SAMPLE.shape[0], EEG_SAMPLE.shape[1]
    SPECTRAL_FEAT = POWER_SAMPLE.shape[0] * POWER_SAMPLE.shape[1]  # F * T_merged
    N_CLASSES = len(torch.unique(torch.tensor(dataset.labels)))
    print(f"Specs: EEG[{EEG_CH}x{EEG_TIME}], Spectral[{SPECTRAL_FEAT}], Classes={N_CLASSES}")
    
    # Конфигурации входов для 5 моделей
    input_configs = {
        1: {'eeg': True, 'power': False, 'phase': False},
        2: {'eeg': True, 'power': True, 'phase': True},
        3: {'eeg': True, 'power': True, 'phase': False},
        4: {'eeg': True, 'power': False, 'phase': True},
        5: {'eeg': False, 'power': True, 'phase': True},
    }
    
    results_storage = {i: [] for i in range(1, 6)}
    subject_ids = sorted(set(rec[0] for rec in dataset.idx_to_record))
    
    # LOSO цикл
    for s_id in subject_ids:
        print(f"\n>>> Subject {s_id} (LOSO)")
        subject_idx = [i for i, rec in enumerate(dataset.idx_to_record) if rec[0] == s_id]
        if len(subject_idx) < min(SPLITS):
            continue
        
        y_subject = np.array([dataset.labels[i].item() if isinstance(dataset.labels[i], torch.Tensor) 
                              else int(dataset.labels[i]) for i in subject_idx])
        
        for n_splits in tqdm(SPLITS, desc='Splits', leave=False):
            k_fold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=TRAIN_CONFIG['seed'])
            
            for fold, (tr_rel, te_rel) in enumerate(k_fold.split(subject_idx, y_subject), 1):
                train_idx = [subject_idx[i] for i in tr_rel]
                test_idx = [subject_idx[i] for i in te_rel]
                
                # === Обучение 5 моделей ===
                for model_id in range(1, 6):
                    cfg = input_configs[model_id]
                    
                    # Инициализация модели (новая на каждый фолд!)
                    model = EIRNet(
                        eeg_ch=EEG_CH,
                        eeg_time=EEG_TIME,
                        spectral_features=SPECTRAL_FEAT,
                        n_classes=N_CLASSES,
                        input_config=cfg
                    ).to(DEVICE)
                    
                    optimizer = optim.AdamW(model.parameters(), 
                                           lr=TRAIN_CONFIG['lr'], 
                                           weight_decay=TRAIN_CONFIG['weight_decay'])
                    criterion = nn.CrossEntropyLoss()
                    scaler = torch.amp.GradScaler(enabled=(DEVICE.type=='cuda'))
                    
                    # DataLoaders
                    collate = lambda b: collate_fn_eir(b, model_id, dataset.eog_needed)
                    train_loader = DataLoader(
                        Subset(dataset, train_idx), 
                        batch_size=TRAIN_CONFIG['batch_size'], 
                        shuffle=True, 
                        collate_fn=collate,
                        num_workers=2,
                        pin_memory=(DEVICE.type=='cuda')
                    )
                    test_loader = DataLoader(
                        Subset(dataset, test_idx), 
                        batch_size=TRAIN_CONFIG['batch_size']*2, 
                        shuffle=False,
                        collate_fn=collate,
                        num_workers=2,
                        pin_memory=(DEVICE.type=='cuda')
                    )
                    
                    # Training loop с early stopping
                    best_f1, patience_counter = 0, 0
                    for epoch in range(TRAIN_CONFIG['epochs']):
                        train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler, model_id, dataset.eog_needed)
                        
                        # Валидация каждые 5 эпох
                        if (epoch + 1) % 5 == 0 or epoch == TRAIN_CONFIG['epochs']-1:
                            metrics = evaluate(model, test_loader, DEVICE, model_id, dataset.eog_needed)
                            if metrics['f1'] > best_f1:
                                best_f1 = metrics['f1']
                                patience_counter = 0
                            else:
                                patience_counter += 1
                            
                            if patience_counter >= TRAIN_CONFIG['early_stop_patience']:
                                break
                    
                    # Финальная оценка
                    final_metrics = evaluate(model, test_loader, DEVICE, model_id, dataset.eog_needed)
                    results_storage[model_id].append({
                        'subject_id': int(s_id),
                        'n_splits': n_splits,
                        'fold': fold,
                        'train_size': len(train_idx),
                        'test_size': len(test_idx),
                        **final_metrics
                    })
                    
                    # Очистка памяти
                    del model, optimizer, train_loader, test_loader
                    torch.cuda.empty_cache()
            
            print(f"  Subject {s_id} done")

    # Сохранение
    RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULT_PATH, 'w', encoding='utf-8') as f:
        json.dump({
            'meta': {
                'device': str(DEVICE),
                'train_config': TRAIN_CONFIG,
                'dataset_config': DATASET_CONFIG,
                'subjects': subject_ids
            },
            'input_configs': {k: str(v) for k,v in input_configs.items()},
            'results': results_storage
        }, f, indent=4, ensure_ascii=False)
    
    print(f"\n✅ Saved to {RESULT_PATH}")

In [12]:
import logging
logging.getLogger('joblib').setLevel(logging.ERROR)
logging.getLogger('mne').setLevel(logging.ERROR)

if __name__ == "__main__":
    main()

Loading dataset from Generated/Data_Train...
Dataset: 1260 samples
Specs: EEG[63x3968], Spectral[3796065], Classes=13

>>> Subject 1 (LOSO)


Splits:  33%|███▎      | 1/3 [18:30<37:00, 1110.04s/it]

  Subject 1 done


Splits:  67%|██████▋   | 2/3 [54:04<28:32, 1712.65s/it]

  Subject 1 done


...

  Subject 4 done

>>> Subject 5 (LOSO)


Splits:  33%|███▎      | 1/3 [18:43<37:26, 1123.46s/it]

  Subject 5 done


  Subject 5 done


ValueError: n_splits=7 cannot be greater than the number of members in each class.